# Compile and evaluate

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/tutorial/core/06_compile_and_evaluate.ipynb)

From a `PlanningProblem` and a controller to a score: `problem.compile(controller)` builds a `ProblemEvaluator`, `pe.evaluate()` returns an `Evaluation` record (traceable under `jax.grad` and batched across Monte Carlo trials), `compare` benchmarks multiple controllers on paired draws, and `sys.compile()` provides the low-level `CompiledEvaluator` underneath.

In [ ]:
# Local conda: minilink already installed. Colab: clone + path + meshcat.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")
    get_ipython().system("pip install -q meshcat")

## 1. `ProblemEvaluator`: compiling a controller on a problem

`pe = problem.compile(controller, dt=None, *, solver="rk4", substeps=None, backend="auto", episode_length=None)` closes `controller` around `problem.sys` (`controller @ plant` for feedback, `controller >> plant` for open-loop sources) and compiles the closed loop once:

| `ProblemEvaluator` Attribute / Method | Arguments | Output type & shape | Meaning |
| --- | --- | --- | --- |
| `pe.problem`, `pe.loop` | — | `PlanningProblem`, `DiagramSystem \| SampledDiagramSystem` | The problem and the closed-loop diagram built by the evaluator |
| `pe.backend`, `pe.dt`, `pe.n_steps` | — | `str`, `float`, `int` | Chosen backend (`"jax"` or `"numpy"`), base period $\Delta t$, and step count |
| `pe.params` | — | `dict[str, Any]` | Unified parameter tree: loop subsystems (`"plant"`, `"controller"`, ...), problem leaves (`"cost"`, `"X"`, `"U"`, `"Xf"`, `"x0"`, `"t0"`, `"discount_rate"`), and free loop inputs |
| `pe.evaluate(params=None, ...)` | `params: dict \| None = None, key=0, n_trials: int \| None = None, *, trajectories: bool = False` | `Evaluation` | Run 1 deterministic trial or `n_trials` Monte Carlo draws and score each run |
| `pe.trajectory(params=None, key=None)` | `params: dict \| None = None, key=None` | `Trajectory` | Closed-loop trajectory of a single run with all block outputs and `"plant"` role recorded |
| `pe.program(wrt, bounds=None, ...)` | `wrt: Sequence[str]`, `bounds: dict \| None = None`, `key=0`, `n_trials=None`, `*, params: dict \| None = None` | `MathematicalProgram` | Export a tuning/co-design NLP over the named parameter paths in `wrt` |
| `pe.pack(params, wrt)` | `params: dict`, `wrt: Sequence[str]` | `ndarray (n_z,)` | Flatten the leaves named by `wrt` into a 1D decision vector $z$ |
| `pe.unpack(z, wrt, params=None)` | `z: array (n_z,)`, `wrt: Sequence[str]`, `params: dict \| None = None` | `dict` | Copy of `params` (or `pe.params`) with the leaves in `wrt` replaced by $z$ |

Standalone functions in `minilink.planning`:
- **`evaluate(problem, controller, *, dt=None, n_trials=None, key=0, trajectories=False, ...) -> Evaluation`**: one-shot compile + evaluate; a sampled controller (`pid % 0.01`) runs on its own clock, `dt` only for a continuous one.
- **`evaluate_trajectory(problem, traj) -> Evaluation`**: score an already-recorded `Trajectory` against `problem`.

In [ ]:
import numpy as np

from minilink import PID, BallSet, InvertedPendulum, PlanningProblem, QuadraticCost

plant = InvertedPendulum()
Q = np.diag([10.0, 1.0])
R = 0.1 * np.eye(1)
cost = QuadraticCost.from_system(plant, Q=Q, R=R)
problem = PlanningProblem(
    plant,
    cost=cost,
    x0=np.array([0.4, 0.0]),
    horizon=4.0,
    Xf=BallSet(np.zeros(2), 0.05),
)

pd = PID(Kp=20.0, Kd=4.0)
pe = problem.compile(pd % 0.01)
pe.backend, pe.dt, pe.n_steps, list(pe.params)

In [ ]:
report = pe.evaluate()
print(report)

`pe.trajectory()` returns the full closed-loop `Trajectory`, and `traj.of("plant")` extracts the plant's own `(t, x, u)` slice:

In [ ]:
traj = pe.trajectory()
plant_traj = traj.of("plant")
traj.signal_names, plant_traj.x.shape, plant_traj.u.shape

In [ ]:
pe.loop.plot_trajectory(traj)

## 2. `Evaluation`: per-run fields and aggregate statistics

`pe.evaluate()` returns an `Evaluation` pytree record containing one entry per trial (`n_trials = 1` on a deterministic run) together with summary reductions:

| `Evaluation` Attribute / Method | Type & shape | Meaning |
| --- | --- | --- |
| `ev.n_trials` | `int` | Number of evaluated runs $K$ |
| `ev.raw_cost` | `array (K,)` | Raw integral + terminal cost $\int e^{-\rho t} g\, dt + e^{-\rho t_f} h(x(t_f))$ (`+inf` if non-finite) |
| `ev.J` | `array (K,)` | Total score (`raw_cost`, plus `failure_cost` when `penalize_failure=True` and the run failed or missed `Xf`) |
| `ev.feasibility_margin` | `array (K,)` | Minimum margin to `X` and `U` along each run ($\ge 0$ means feasible) |
| `ev.failed` | `bool array (K,)` | `True` when a run overflowed or had `feasibility_margin < 0` |
| `ev.reachability_margin` | `array (K,)` | Terminal margin of $x(t_f)$ to `Xf` (`+inf` when `Xf is None`) |
| `ev.reached` | `bool array (K,)` | `True` when `reachability_margin >= 0` |
| `ev.x0` | `array (K, n)` | Initial state $x(t_0)$ of each trial |
| `ev.trajectories` | `list[Trajectory] \| None` | Per-trial trajectories when `pe.evaluate(trajectories=True)` |
| `ev.objective` | `float` (scalar) | Criterion reduction of `J` (`mean`, `worst`, or `cvar(alpha)`) minimized by optimizers |
| `ev.mean`, `ev.std`, `ev.worst` | `float` (scalar) | Sample mean, standard deviation, and maximum of `J` across the $K$ trials |
| `ev.cvar(alpha)` | `alpha: float in (0, 1] -> float` | Mean of the worst $\lceil \alpha K \rceil$ values of `J` |
| `ev.feasibility_rate`, `ev.reachability_rate`, `ev.success_rate` | `float` in $[0, 1]$ | Fraction of runs that stayed feasible, reached `Xf`, or achieved both |

In [ ]:
(
    report.cost,
    report.J,
    report.feasibility_margin,
    report.failed,
    report.reachability_margin,
    report.reached,
    report.x0,
)

In [ ]:
# Other numbers without recompiling: the evaluator's tree (pe.params is a copy), edited
heavy = pe.params
heavy["plant"]["m"] = 1.2
heavier = pe.evaluate(heavy)
report.objective, heavier.objective

## 3. Differentiating `pe.evaluate(params).objective` with JAX

Because `ProblemEvaluator` and `Evaluation` are JAX-traceable, `jax.grad` differentiates `pe.evaluate(params).objective` with respect to every leaf of `pe.params` in a single backward pass:

In [ ]:
import jax

dJ = jax.grad(lambda params: pe.evaluate(params).objective)(pe.params)
dJ["controller"]["Kp"], dJ["controller"]["Kd"], dJ["plant"]["m"]

## 4. Monte Carlo evaluation on a stochastic problem

Passing `n_trials=200, key=1` draws 200 initial states and parameter realizations and evaluates the batch in one vectorized call:

In [ ]:
from minilink import Uniform

random_problem = problem.replace(
    x0=Uniform([0.2, -0.2], [0.6, 0.2]),
    distributions={"plant": {"m": Uniform([0.8], [1.2])}},
)
monte_carlo = random_problem.compile(pd % 0.01).evaluate(n_trials=200, key=1)
print(monte_carlo)

In [ ]:
(
    monte_carlo.n_trials,
    monte_carlo.mean,
    monte_carlo.std,
    monte_carlo.worst,
    monte_carlo.cvar(0.1),
    monte_carlo.feasibility_rate,
    monte_carlo.reachability_rate,
    monte_carlo.success_rate,
)

## 5. Benchmarking controllers side by side with `compare`

`compare(problem, Name=controller_or_solution, ..., dt=None, n_trials=None, key=0)` evaluates multiple controllers on the exact same draws and returns a `Comparison` object; `dt` as in `evaluate` (sampled controllers run on their own clock, a continuous one needs `dt`):
- `print(comparison)` prints a summary table;
- `comparison.evaluations[name]` holds each controller's `Evaluation`;
- `comparison.difference(name_a, name_b)` returns the paired per-trial cost difference $J_a - J_b$ of shape `(n_trials,)`.

In [ ]:
from minilink import compare
from minilink.control import lqr

comparison = compare(
    random_problem,
    LQR=lqr(plant, Q, R) % 0.01,
    PD=pd % 0.01,
    PID=PID(Kp=20.0, Ki=10.0, Kd=4.0) % 0.01,
    n_trials=50,
    key=1,
)
print(comparison)

In [ ]:
diff_pd_lqr = comparison.difference("PD", "LQR")
diff_pd_lqr.shape, diff_pd_lqr.mean()

## 6. Under the hood: `sys.compile(backend=...)` and `CompiledEvaluator`

`sys.compile(backend="numpy" | "jax")` lowers any `System` (leaf or diagram, continuous or discrete) into a flat `CompiledEvaluator` with zero object-dispatch overhead:

| `CompiledEvaluator` Method | Arguments | Output type & shape | Meaning |
| --- | --- | --- | --- |
| `ev.f(x, u, t=0.0, params=None)` | `x: (n,)`, `u: (m,)`, `t: float = 0.0`, `params=None` | `array (n,)` | Continuous state derivative $\dot x$ |
| `ev.outputs(x, u, t=0.0, params=None)` | `x: (n,)`, `u: (m,)`, `t: float = 0.0`, `params=None` | `dict[str, array (dim,)]` | Dictionary of all boundary output ports (e.g. `outputs["y"]`) |
| `ev.step(x, u, t, params=None, ...)` | `x: (n,)`, `u: (m,)`, `t: float`, `params=None, *, dt, solver="rk4", substeps=1` | `array (n,)` | State one period `dt` later with input `u` held |
| `ev.rollout(x0, u_seq, t0, dt, params=None, ...)` | `x0: (n,)`, `u_seq: (N, m) \| None`, `t0: float`, `dt: float`, `params=None, *, solver="rk4", substeps=1, hold="zoh", n_steps=None` | `array (N + 1, n)` | Multi-step rollout over $N$ intervals (`lax.scan` on JAX) |
| `ev.rollout_batch(x0s, u_sequences=None, ...)` | `x0s: (B, n)`, `u_sequences: (B, N, m) \| None = None`, `*, t0=0.0, dt, n_steps=None, params=None` | `array (B, N + 1, n)` | Batched rollout across $B$ initial states (`vmap` on JAX) |
| `ev.jacobian(of, wrt, x, u, t=0.0, params=None)` | `of: "f" \| "step" \| port_id`, `wrt: "x" \| "u" \| "t" \| "params"`, the point | `array` | $\partial(\text{of})/\partial(\text{wrt})$ at the point, as `sys.jacobian`; the function form is the method, `partial(ev.jacobian, "f", "x")` |

In [ ]:
evaluator = plant.compile(backend="jax")

x = np.array([0.4, 0.0])
u = np.array([0.0])
(
    evaluator.f(x, u),
    evaluator.outputs(x, u),
    evaluator.step(x, u, 0.0, dt=0.01),
)

In [ ]:
from functools import partial

df_dx = partial(evaluator.jacobian, "f", "x")  # the derivative as a function
df_dx(x, u, 0.0, None)

In [ ]:
u_seq = np.zeros((100, 1))
x_path = evaluator.rollout(x, u_seq, 0.0, 0.01)

x0s = np.array([[0.1, 0.0], [0.2, 0.0], [0.3, 0.0]])
x_paths = evaluator.rollout_batch(x0s, dt=0.01, n_steps=100)
x_path.shape, x_paths.shape

Next, [`07_program`](07_program.ipynb): `MathematicalProgram`, `Optimizer`, and `OptimizationResult`.